# AAT6153 Chapter 2 — Hands-on 1
## Text Vectorization: from Raw Text to Integers
**Slides:** Working with Text Data (3–4), Text Vectorization (5–10), Hands-on 1 (11–14)

Deep learning models only process numbers, so text must first be **vectorized**:

**raw text → standardization → tokenization → indexing → encoding**

In this notebook you will:
1. Implement the pipeline yourself in plain Python (slide 11).
2. Do the same with the Keras `TextVectorization` layer (slides 12–13).
3. Build a **bag-of-words** (multi-hot) representation with **bigrams** (slide 14).
4. **Pad and truncate** sequences to a common length (slide 10).

*Runtime: CPU is fine for this notebook.*

---
## Step 1: The pipeline in plain Python (slide 11)
Indices 0 and 1 are reserved: **0 = padding**, **1 = out-of-vocabulary (OOV)**. Real words start at 2.

In [ ]:
text = "The cat sat on the mat. The dog sat on the log!"

# 1. Standardize: lowercase and strip punctuation
clean = "".join(c for c in text.lower() if c.isalnum() or c == " ")
# 2. Tokenize: split on whitespace
tokens = clean.split()
# 3. Index: 0 = padding, 1 = out-of-vocabulary (OOV)
vocab = {w: i + 2 for i, w in enumerate(sorted(set(tokens)))}
# 4. Encode the tokens as integers
encoded = [vocab.get(w, 1) for w in tokens]

print(tokens)
print(vocab)
print(encoded)

In [ ]:
# Encode a new sentence: unseen words map to 1 (OOV)
new = "the bird sat on the cat"
print([vocab.get(w, 1) for w in new.split()])

## Step 2: Build a vocabulary with the `TextVectorization` layer (slide 12)
`TextVectorization` performs standardization, tokenization and indexing in one layer. `adapt()` scans the corpus and builds the vocabulary, **sorted by frequency**.

In [ ]:
import keras
from keras import layers

corpus = ["I love deep learning!",
          "Deep learning loves data.",
          "I love Keras and Python."]

vectorizer = layers.TextVectorization(
    max_tokens=20,              # maximum vocabulary size
    output_mode="int",          # map each token to an integer
    output_sequence_length=6)   # pad / truncate to 6 tokens
vectorizer.adapt(corpus)        # learn the vocabulary

print([str(w) for w in vectorizer.get_vocabulary()])

## Step 3: Encode new sentences (slide 13)

In [ ]:
print(vectorizer(["I love Python and deep learning"]))
print(vectorizer(["I love TensorFlow"]))

"tensorflow" is not in the vocabulary, so it becomes **1 ([UNK])**, and the rest is padded with **0**.

**Try it:** what happens to a sentence with more than 6 words?

In [ ]:
print(vectorizer(["I love deep learning and I love Keras and Python too"]))

## Step 4: Bag-of-words (multi-hot) with bigrams (slides 8, 14)
`multi_hot` gives one position per vocabulary entry: **1 if the token is present**, regardless of order. With `ngrams=2` the vocabulary also contains bigrams such as *deep learning*.

In [ ]:
bow = layers.TextVectorization(
    max_tokens=20,
    output_mode="multi_hot",    # bag-of-words: 1 if token present
    ngrams=2)                   # unigrams + bigrams
bow.adapt(corpus)

print([str(w) for w in bow.get_vocabulary()])
print(bow(["I love deep learning"]))

In [ ]:
# Bag-of-words ignores word order (slide 4): these two sentences get the same vector
unigram_bow = layers.TextVectorization(output_mode="multi_hot")
unigram_bow.adapt(["dog bites man"])
print(unigram_bow(["dog bites man", "man bites dog"]))

## Step 5: Padding and truncation (slide 10)
Every sequence in a batch must have the same length. `pad_sequences` adds zeros and cuts long sequences.

In [ ]:
from keras.utils import pad_sequences

seqs = [[5, 2, 8], [3, 9, 4, 7, 1, 6], [2]]
print(pad_sequences(seqs, maxlen=4))                    # pre-padding
print(pad_sequences(seqs, maxlen=4, padding="post",
                    truncating="post"))                 # post-padding

---
## Try it yourself
1. In Step 4, change `output_mode` to `"count"` and then `"tf_idf"`. How do the vectors differ from `multi_hot`?
2. In Step 2, set `standardize=None`. Why does `"learning!"` now become a different token from `"learning"`? (slide 6)
3. Set `max_tokens=6` in Step 2. Which words end up as `[UNK]`?
4. Add `split="character"` to a new `TextVectorization` and adapt it on the corpus. How big is the vocabulary now? (slide 7)

In [ ]:
# Use this cell for your experiments